In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    regexp_replace,
    explode,
    concat_ws,
    when,
    count
)
from pyspark.sql.types import *

In [7]:
spark = (
    SparkSession.builder
    .appName("Gaming Analytics - RAWG Data Cleaning")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.local.dir", "D:/spark_temp")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [8]:
print("\n========== LOAD RAWG BRONZE DATA ==========\n")

rawg = (
    spark.read
    .option(
        "multiline",
        "true"
    )
    .json(
        "../data/bronze/rawg_games.json"
    )
)

print("RAWG rows:")
print(rawg.count())

rawg.printSchema()


========== LOAD RAWG BRONZE DATA ==========

RAWG rows:
760
root
 |-- added: long (nullable = true)
 |-- added_by_status: struct (nullable = true)
 |    |-- beaten: long (nullable = true)
 |    |-- dropped: long (nullable = true)
 |    |-- owned: long (nullable = true)
 |    |-- playing: long (nullable = true)
 |    |-- toplay: long (nullable = true)
 |    |-- yet: long (nullable = true)
 |-- background_image: string (nullable = true)
 |-- clip: string (nullable = true)
 |-- dominant_color: string (nullable = true)
 |-- esrb_rating: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- slug: string (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- games_count: long (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- image_background: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- slug: string (nulla

In [9]:
print(rawg.columns)

rawg.printSchema()

['added', 'added_by_status', 'background_image', 'clip', 'dominant_color', 'esrb_rating', 'genres', 'id', 'metacritic', 'name', 'parent_platforms', 'platforms', 'playtime', 'rating', 'rating_top', 'ratings', 'ratings_count', 'released', 'reviews_count', 'reviews_text_count', 'saturated_color', 'short_screenshots', 'slug', 'stores', 'suggestions_count', 'tags', 'tba', 'updated', 'user_game']
root
 |-- added: long (nullable = true)
 |-- added_by_status: struct (nullable = true)
 |    |-- beaten: long (nullable = true)
 |    |-- dropped: long (nullable = true)
 |    |-- owned: long (nullable = true)
 |    |-- playing: long (nullable = true)
 |    |-- toplay: long (nullable = true)
 |    |-- yet: long (nullable = true)
 |-- background_image: string (nullable = true)
 |-- clip: string (nullable = true)
 |-- dominant_color: string (nullable = true)
 |-- esrb_rating: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- slug: string (

In [10]:
print("\n========== SELECT REQUIRED COLUMNS ==========\n")


rawg = rawg.select(
    col("id").alias("rawg_id"),
    col("name"),
    col("rating"),
    col("ratings_count"),
    col("metacritic"),
    col("genres"),
    col("platforms")
)

rawg.printSchema()


========== SELECT REQUIRED COLUMNS ==========

root
 |-- rawg_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- ratings_count: long (nullable = true)
 |-- metacritic: long (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- games_count: long (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- image_background: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- slug: string (nullable = true)
 |-- platforms: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- platform: struct (nullable = true)
 |    |    |    |-- games_count: long (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: string (nullable = true)
 |    |    |    |-- image_background: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-

In [11]:
print("\n========== REMOVE MISSING GAME NAMES ==========\n")


rawg = rawg.dropna(
    subset=[
        "name"
    ]
)


print("DONE")


========== REMOVE MISSING GAME NAMES ==========

DONE


In [12]:
print("\n========== CREATE CLEAN GAME NAME ==========\n")


rawg = rawg.withColumn(
    "game_name_clean",
    lower(
        trim(
            regexp_replace(
                col("name"),
                "[^a-zA-Z0-9 ]",
                ""
            )
        )
    )
)


print("DONE")


========== CREATE CLEAN GAME NAME ==========

DONE


In [13]:
print("\n========== EXTRACT GENRES ==========\n")


rawg = rawg.withColumn(
    "genres",
    concat_ws(
        ", ",
        col("genres.name")
    )
)


print("DONE")


========== EXTRACT GENRES ==========

DONE


In [14]:
print("\n========== EXTRACT PLATFORMS ==========\n")


rawg = rawg.withColumn(
    "platforms",
    concat_ws(
        ", ",
        col("platforms.platform.name")
    )
)


print("DONE")


========== EXTRACT PLATFORMS ==========

DONE


In [15]:
print("\n========== CONVERT DATA TYPES ==========\n")


rawg = rawg.withColumn(
    "rating",
    col("rating").cast("double")
)


rawg = rawg.withColumn(
    "ratings_count",
    col("ratings_count").cast("integer")
)


rawg = rawg.withColumn(
    "metacritic",
    col("metacritic").cast("integer")
)


print("DONE")


========== CONVERT DATA TYPES ==========

DONE


In [16]:
print("\n========== HANDLE MISSING VALUES ==========\n")


rawg = rawg.fillna(
    {
        "ratings_count": 0,
        "metacritic": 0
    }
)


print("DONE")


========== HANDLE MISSING VALUES ==========

DONE


In [17]:
print("\n========== FINAL RAWG SILVER PREVIEW ==========\n")


rawg.show(
    5,
    truncate=False
)


rawg.printSchema()


========== FINAL RAWG SILVER PREVIEW ==========

+-------+--------------------------------+------+-------------+----------+---------------+------------------------------------------------------------------------------------+------------------------------+
|rawg_id|name                            |rating|ratings_count|metacritic|genres         |platforms                                                                           |game_name_clean               |
+-------+--------------------------------+------+-------------+----------+---------------+------------------------------------------------------------------------------------+------------------------------+
|3498   |Grand Theft Auto V              |4.47  |7419         |92        |Action         |PlayStation 5, Xbox Series S/X, PlayStation 3, PC, PlayStation 4, Xbox 360, Xbox One|grand theft auto v            |
|3328   |The Witcher 3: Wild Hunt        |4.64  |7241         |92        |Action, RPG    |PlayStation 5, Xbox Series S/X, 

In [18]:
print("\n========== RAWG DATA QUALITY CHECK ==========\n")


print("Rows:")
print(rawg.count())


print("\nNull Values:")


rawg.select([
    count(
        when(
            col(c).isNull(),
            c
        )
    ).alias(c)
    for c in rawg.columns
]).show()


========== RAWG DATA QUALITY CHECK ==========

Rows:
760

Null Values:
+-------+----+------+-------------+----------+------+---------+---------------+
|rawg_id|name|rating|ratings_count|metacritic|genres|platforms|game_name_clean|
+-------+----+------+-------------+----------+------+---------+---------------+
|      0|   0|     0|            0|         0|     0|        0|              0|
+-------+----+------+-------------+----------+------+---------+---------------+



In [19]:
print("\n========== SAVE RAWG SILVER DATA ==========\n")


rawg.write \
    .mode("overwrite") \
    .parquet(
        "../data/silver/rawg_games_clean"
    )


print("RAWG Silver saved successfully")


========== SAVE RAWG SILVER DATA ==========

RAWG Silver saved successfully


In [20]:
spark.stop()

print("RAWG cleaning completed successfully")

RAWG cleaning completed successfully
